# Training MobileNetV3-Large (Moderasi Satu Data)
Fine-tune **MobileNetV3-Large** untuk 3 kelas: `dokumen` / `normal` / `obat_aborsi`.

**Alur:**
1. Zip dataset di Windows, upload ke Google Drive.
2. Jalankan cell di bawah secara berurutan (Ctrl+Enter / Shift+Enter).
3. Download `mobilenetv3_best.pt`, taruh di folder `models/` proyek.

## Langkah 0 — Zip dataset di Windows
Buka PowerShell di folder proyek ini, lalu:
```powershell
Compress-Archive -Path dataset -DestinationPath dataset.zip
```
Upload **`dataset.zip`** ke Google Drive (mis. `MyDrive`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
from google.colab import files

# (a) Upload train_mobilenetv3.py dari komputer (lihat tombol folder di panel kiri Colab)
assert os.path.isfile('/content/train_mobilenetv3.py'), "Upload train_mobilenetv3.py dulu!"

# (b) Pilih sumber dataset.zip:
#     Opsi 1: dari Google Drive
ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
#     Opsi 2: upload langsung dari komputer (hapus '# ' di baris berikut, lalu pilih file)
# uploaded = files.upload(); ZIP_PATH = list(uploaded.keys())[0]

assert os.path.isfile(ZIP_PATH), f'dataset.zip tidak ditemukan: {ZIP_PATH}'
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content')

ROOT = '/content/dataset' if os.path.isdir('/content/dataset') else '/content'
assert os.path.isdir(os.path.join(ROOT, 'train')) and os.path.isdir(os.path.join(ROOT, 'val')), \
    'Struktur zip harus berisi dataset/train dan dataset/val'
print('OK. train:', sorted(os.listdir(os.path.join(ROOT, 'train'))))
print('    val  :', sorted(os.listdir(os.path.join(ROOT, 'val'))))

In [ ]:
EPOCHS = 20   # sesuaikan: makin besar makin lama tapi biasanya makin akurat
BATCH  = 64   # GPU T4 aman di 64

!python /content/train_mobilenetv3.py \
    --epochs $EPOCHS \
    --batch $BATCH \
    --device cuda:0 \
    --data $ROOT \
    --out /content/mobilenetv3_best.pt

In [ ]:
from google.colab import files
# Simpan salinan ke Google Drive
!cp /content/mobilenetv3_best.pt '/content/drive/MyDrive/mobilenetv3_best.pt'
# Unduh ke komputer lokal (taruh di folder proyek > models/)
files.download('/content/mobilenetv3_best.pt')

## Langkah berikutnya di komputer
1. Letakkan hasil unduhan sebagai **`models/mobilenetv3_best.pt`** di folder proyek (ganti file lama kalau ada).
2. Uji:
   ```powershell
   uv run python moderasi.py "path_gambar.jpg" --visual mobilenetv3
   ```
3. Commit & push supaya model ikut ter-deploy ke VPS (deploy.sh sudah otomatis menyalinnya).